# ADEPT compact FL analysis: 10 clients, CIFAR-10/CNN 16k

RQ1 describes the observed objective tradeoffs across configurations and pattern counts. RQ2 searches for operating conditions associated with each observed tradeoff and evaluates the rules on held-out partition seeds. Results are observational associations. Dataset and model are treated as a paired scenario context.

In [ ]:
import sys, contextlib, io, itertools
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append('..')
from adept import PatternAnalysis
from adept.utils.spec_summary import summarize_system
from adept.utils.linter import SystemLinter
import fl_slices

## Common environment

This view uses the same fixed objective bin edges as every other scenario. The common-environment view compares runs at environmental settings shared across configurations. The all-environment view includes all available settings; configuration differences can also reflect differences in sampled environments. These are nested views of the campaign, not independent replications.

In [ ]:
SLICE, INCLUDE_BASELINE = 'all', False
CLIENTS, TASK, ENVIRONMENT = 10, 'image', 'common'
SPEC = './compact/FLsystem_compact_all.json'
TRADEOFF_LABELS = {'final_val_accuracy': ['S', 'M', 'L'], 'avg_total_time': ['S', 'M', 'L']}
TRADEOFF_EDGES = {'final_val_accuracy': [0.2075, 0.4774, 0.7473, 1.0172], 'avg_total_time': [1.5625, 9.5065, 17.4505, 25.3945]}
session = PatternAnalysis(SPEC)
session.load(preprocessor=fl_slices.slice_filter(SLICE, INCLUDE_BASELINE, clients=CLIENTS, task=TASK, environment=ENVIRONMENT))
print(f'Loaded {len(session.raw_df)} experiments')
print(summarize_system(session.sys_def))
issues = SystemLinter().lint(session.sys_def, session.raw_df)
errors = [i for i in issues if i.level == 'ERROR']
warnings = [i for i in issues if i.level == 'WARNING']
print(f'Linter: {len(errors)} errors, {len(warnings)} warnings')
assert not errors

### RQ1 · Which tradeoffs occur, and how do decisions relate to them?

In [ ]:
tradeoffs, schemes = session.create_tradeoffs(method='discretization', labels=TRADEOFF_LABELS, edges=TRADEOFF_EDGES)
for scheme in schemes:
    print(scheme.objective_name, [scheme.bins[0].min_value] + [b.max_value for b in scheme.bins])
print(f'{len(tradeoffs)} defined regions; {len(session.get_tradeoffs(non_empty=True))} contain observations in this scenario/view')
session.split_data(test_size=0.3, random_state=42, remove_outliers=False, verbose=True, group_column='partition_seed')

**Reading the objective-space chart.** Each point is one experimental run; axes show measured outcomes, color identifies the number of active pattern types, and the shared grid defines S/M/L tradeoff quadrants. Clustering shows observed associations; it does not isolate a pattern’s causal effect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
patterns = session.experiments_df
for ax, (x, y) in zip(axes, itertools.combinations(TRADEOFF_LABELS, 2)):
    view = session.raw_df.copy()
    view['Pattern Types Active'] = session.experiments_df['Pattern Types Active'].to_numpy()
    view['Pattern Count Group'] = view['Pattern Types Active'].map(lambda n: '0' if n == 0 else ('1' if n == 1 else '2-3'))
    for level, group in view.groupby('Pattern Count Group', sort=False):
        ax.scatter(group[x], group[y], s=22, alpha=.65, label=f'{level} active pattern types')
    ax.set(xlabel=x, ylabel=y, title=f'{x} vs {y}')
    for edge in TRADEOFF_EDGES[x][1:-1]: ax.axvline(edge, color='black', linewidth=.8, alpha=.55)
    for edge in TRADEOFF_EDGES[y][1:-1]: ax.axhline(edge, color='black', linewidth=.8, alpha=.55)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Tradeoff frequency by decision.** Rows show the share of runs in each fixed region for each complete configuration and each pattern-count group (0, 1, 2–3). These conditional frequencies describe how the observed design alternatives are distributed.

In [ ]:
discrete = session.discrete_df.astype(str).agg('-'.join, axis=1).rename('tradeoff')
view = session.experiments_df.copy()
view['tradeoff'] = discrete.to_numpy()
view['Pattern Count Group'] = view['Pattern Types Active'].map(lambda n: '0' if n == 0 else ('1' if n == 1 else '2-3'))
config_col = session.sys_def.dataspace.configuration_identification.column
for group_col in (config_col, 'Pattern Count Group'):
    counts = pd.crosstab(view[group_col], view['tradeoff'])
    shares = counts.div(counts.sum(axis=1), axis=0)
    print(f'Observed tradeoff counts by {group_col}:'); display(counts)
    print(f'Within-group shares by {group_col}:'); display(shares.round(3))

### RQ2 · Which conditions are associated with each tradeoff, and what changes inside the discovered box?

Feature importance uses ADEPT’s existing train-set ranking to prioritize candidate predictors. It screens variables for scenario discovery; it is not a causal effect estimate.

In [ ]:
scores_df = session.compute_feature_scores(use_smart_correlation=True, include_constraints=True, subset='train')
session.show_feature_heatmap(scores_df, title='Train-set feature importance', figsize=(5, 8)); plt.show()
weighted_features = session.get_weighted_feature_ranking(scores_df)
key_parameters = session.select_top_k_parameters(scores_df, index_order=weighted_features, threshold=0.03, k=None)
print('Weighted ranking:', weighted_features)
print('Candidate parameters (importance threshold 0.03):', key_parameters)

PRIM and CART are fitted on the training partition. Empty quadrants remain in the fixed grid but are not discovery targets. A missing box means that method did not return a nonempty rule for that target.

In [ ]:
prim_boxes = []
for tradeoff in session.get_tradeoffs(non_empty=True):
    boxes = session.discover_scenarios(tradeoff.name, method='prim', threshold=0.8, standardize=True, parameters=key_parameters)
    found = [b for b in boxes if not b.is_empty()]
    if found: prim_boxes.append(found[0])
    else: print('PRIM found no box:', tradeoff.name)
cart_boxes = [b for b in session.discover_scenarios(method='cart', standardize=True, parameters=key_parameters) if not b.is_empty()]
for method, boxes in [('PRIM', prim_boxes), ('CART', cart_boxes)]:
    print(f'\n{method}: {len(boxes)} boxes')
    for i, box in enumerate(boxes, 1):
        print(i, box.target_tradeoff, box.target_tradeoff_labels, box.metrics)
        for parameter, condition in box.describe_limits(digits=3).items(): print('  ', parameter, condition)

**Architecture context for box predicates.** The explanation expands specification-declared parent/child relationships to make interacting policy conditions readable. The original box predicates and thresholds remain the analytic rules.

In [ ]:
for method, boxes in [('PRIM', prim_boxes), ('CART', cart_boxes)]:
    for i, box in enumerate(boxes, 1):
        explanation = session.explain_box_context(box, output_format='plain')
        print(f'{method} box {i}: {box.target_tradeoff}\n{explanation["text"]}\n')

**Held-out what-if comparison.** For each target, compare its baseline frequency in the test partition with its frequency among test runs satisfying the box. Also report box coverage (fraction of all target runs captured), precision (target fraction inside the box), lift (precision minus baseline prevalence), and counts. These are observed conditional differences, not causal improvements. Small or zero test support makes estimates unstable or unavailable.

In [ ]:
def heldout_box_summary(boxes, method):
    X, _, discrete_test = session._get_subset_data('test')
    X = session._encode_for_boxes(X)
    rows = []
    for box in boxes:
        mask = pd.Series(True, index=X.index)
        for parameter, bounds in box.limits.items():
            if parameter in X.columns: mask &= X[parameter].between(bounds['min'], bounds['max'])
        target = pd.Series(True, index=X.index)
        target_definition = session.get_tradeoff(box.target_tradeoff)
        for objective, label in target_definition.elements.items(): target &= session.discrete_df.loc[X.index, objective].astype(str).eq(str(label))
        baseline_n, target_n = len(X), int(target.sum())
        inside_n, inside_target = int(mask.sum()), int((mask & target).sum())
        baseline = target_n / baseline_n if baseline_n else float('nan')
        precision = inside_target / inside_n if inside_n else float('nan')
        coverage = inside_target / target_n if target_n else float('nan')
        rows.append({'method':method, 'target':box.target_tradeoff, 'test_n':baseline_n, 'baseline_target_n':target_n, 'box_n':inside_n, 'target_in_box':inside_target, 'baseline_rate':baseline, 'box_rate_precision':precision, 'test_delta_rate':precision-baseline if inside_n else float('nan'), 'train_lift_absolute':box.metrics.get('lift'), 'coverage_recall':coverage})
    return pd.DataFrame(rows)
what_if = pd.concat([heldout_box_summary(prim_boxes, 'PRIM'), heldout_box_summary(cart_boxes, 'CART')], ignore_index=True)
display(what_if.round(3))

## All environments

This view uses the same fixed objective bin edges as every other scenario. The common-environment view compares runs at environmental settings shared across configurations. The all-environment view includes all available settings; configuration differences can also reflect differences in sampled environments. These are nested views of the campaign, not independent replications.

In [ ]:
SLICE, INCLUDE_BASELINE = 'all', False
CLIENTS, TASK, ENVIRONMENT = 10, 'image', 'all'
SPEC = './compact/FLsystem_compact_all.json'
TRADEOFF_LABELS = {'final_val_accuracy': ['S', 'M', 'L'], 'avg_total_time': ['S', 'M', 'L']}
TRADEOFF_EDGES = {'final_val_accuracy': [0.2075, 0.4774, 0.7473, 1.0172], 'avg_total_time': [1.5625, 9.5065, 17.4505, 25.3945]}
session = PatternAnalysis(SPEC)
session.load(preprocessor=fl_slices.slice_filter(SLICE, INCLUDE_BASELINE, clients=CLIENTS, task=TASK, environment=ENVIRONMENT))
print(f'Loaded {len(session.raw_df)} experiments')
print(summarize_system(session.sys_def))
issues = SystemLinter().lint(session.sys_def, session.raw_df)
errors = [i for i in issues if i.level == 'ERROR']
warnings = [i for i in issues if i.level == 'WARNING']
print(f'Linter: {len(errors)} errors, {len(warnings)} warnings')
assert not errors

### RQ1 · Which tradeoffs occur, and how do decisions relate to them?

In [ ]:
tradeoffs, schemes = session.create_tradeoffs(method='discretization', labels=TRADEOFF_LABELS, edges=TRADEOFF_EDGES)
for scheme in schemes:
    print(scheme.objective_name, [scheme.bins[0].min_value] + [b.max_value for b in scheme.bins])
print(f'{len(tradeoffs)} defined regions; {len(session.get_tradeoffs(non_empty=True))} contain observations in this scenario/view')
session.split_data(test_size=0.3, random_state=42, remove_outliers=False, verbose=True, group_column='partition_seed')

**Reading the objective-space chart.** Each point is one experimental run; axes show measured outcomes, color identifies the number of active pattern types, and the shared grid defines S/M/L tradeoff quadrants. Clustering shows observed associations; it does not isolate a pattern’s causal effect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
patterns = session.experiments_df
for ax, (x, y) in zip(axes, itertools.combinations(TRADEOFF_LABELS, 2)):
    view = session.raw_df.copy()
    view['Pattern Types Active'] = session.experiments_df['Pattern Types Active'].to_numpy()
    view['Pattern Count Group'] = view['Pattern Types Active'].map(lambda n: '0' if n == 0 else ('1' if n == 1 else '2-3'))
    for level, group in view.groupby('Pattern Count Group', sort=False):
        ax.scatter(group[x], group[y], s=22, alpha=.65, label=f'{level} active pattern types')
    ax.set(xlabel=x, ylabel=y, title=f'{x} vs {y}')
    for edge in TRADEOFF_EDGES[x][1:-1]: ax.axvline(edge, color='black', linewidth=.8, alpha=.55)
    for edge in TRADEOFF_EDGES[y][1:-1]: ax.axhline(edge, color='black', linewidth=.8, alpha=.55)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Tradeoff frequency by decision.** Rows show the share of runs in each fixed region for each complete configuration and each pattern-count group (0, 1, 2–3). These conditional frequencies describe how the observed design alternatives are distributed.

In [ ]:
discrete = session.discrete_df.astype(str).agg('-'.join, axis=1).rename('tradeoff')
view = session.experiments_df.copy()
view['tradeoff'] = discrete.to_numpy()
view['Pattern Count Group'] = view['Pattern Types Active'].map(lambda n: '0' if n == 0 else ('1' if n == 1 else '2-3'))
config_col = session.sys_def.dataspace.configuration_identification.column
for group_col in (config_col, 'Pattern Count Group'):
    counts = pd.crosstab(view[group_col], view['tradeoff'])
    shares = counts.div(counts.sum(axis=1), axis=0)
    print(f'Observed tradeoff counts by {group_col}:'); display(counts)
    print(f'Within-group shares by {group_col}:'); display(shares.round(3))

### RQ2 · Which conditions are associated with each tradeoff, and what changes inside the discovered box?

Feature importance uses ADEPT’s existing train-set ranking to prioritize candidate predictors. It screens variables for scenario discovery; it is not a causal effect estimate.

In [ ]:
scores_df = session.compute_feature_scores(use_smart_correlation=True, include_constraints=True, subset='train')
session.show_feature_heatmap(scores_df, title='Train-set feature importance', figsize=(5, 8)); plt.show()
weighted_features = session.get_weighted_feature_ranking(scores_df)
key_parameters = session.select_top_k_parameters(scores_df, index_order=weighted_features, threshold=0.03, k=None)
print('Weighted ranking:', weighted_features)
print('Candidate parameters (importance threshold 0.03):', key_parameters)

PRIM and CART are fitted on the training partition. Empty quadrants remain in the fixed grid but are not discovery targets. A missing box means that method did not return a nonempty rule for that target.

In [ ]:
prim_boxes = []
for tradeoff in session.get_tradeoffs(non_empty=True):
    boxes = session.discover_scenarios(tradeoff.name, method='prim', threshold=0.8, standardize=True, parameters=key_parameters)
    found = [b for b in boxes if not b.is_empty()]
    if found: prim_boxes.append(found[0])
    else: print('PRIM found no box:', tradeoff.name)
cart_boxes = [b for b in session.discover_scenarios(method='cart', standardize=True, parameters=key_parameters) if not b.is_empty()]
for method, boxes in [('PRIM', prim_boxes), ('CART', cart_boxes)]:
    print(f'\n{method}: {len(boxes)} boxes')
    for i, box in enumerate(boxes, 1):
        print(i, box.target_tradeoff, box.target_tradeoff_labels, box.metrics)
        for parameter, condition in box.describe_limits(digits=3).items(): print('  ', parameter, condition)

**Architecture context for box predicates.** The explanation expands specification-declared parent/child relationships to make interacting policy conditions readable. The original box predicates and thresholds remain the analytic rules.

In [ ]:
for method, boxes in [('PRIM', prim_boxes), ('CART', cart_boxes)]:
    for i, box in enumerate(boxes, 1):
        explanation = session.explain_box_context(box, output_format='plain')
        print(f'{method} box {i}: {box.target_tradeoff}\n{explanation["text"]}\n')

**Held-out what-if comparison.** For each target, compare its baseline frequency in the test partition with its frequency among test runs satisfying the box. Also report box coverage (fraction of all target runs captured), precision (target fraction inside the box), lift (precision minus baseline prevalence), and counts. These are observed conditional differences, not causal improvements. Small or zero test support makes estimates unstable or unavailable.

In [ ]:
def heldout_box_summary(boxes, method):
    X, _, discrete_test = session._get_subset_data('test')
    X = session._encode_for_boxes(X)
    rows = []
    for box in boxes:
        mask = pd.Series(True, index=X.index)
        for parameter, bounds in box.limits.items():
            if parameter in X.columns: mask &= X[parameter].between(bounds['min'], bounds['max'])
        target = pd.Series(True, index=X.index)
        target_definition = session.get_tradeoff(box.target_tradeoff)
        for objective, label in target_definition.elements.items(): target &= session.discrete_df.loc[X.index, objective].astype(str).eq(str(label))
        baseline_n, target_n = len(X), int(target.sum())
        inside_n, inside_target = int(mask.sum()), int((mask & target).sum())
        baseline = target_n / baseline_n if baseline_n else float('nan')
        precision = inside_target / inside_n if inside_n else float('nan')
        coverage = inside_target / target_n if target_n else float('nan')
        rows.append({'method':method, 'target':box.target_tradeoff, 'test_n':baseline_n, 'baseline_target_n':target_n, 'box_n':inside_n, 'target_in_box':inside_target, 'baseline_rate':baseline, 'box_rate_precision':precision, 'test_delta_rate':precision-baseline if inside_n else float('nan'), 'train_lift_absolute':box.metrics.get('lift'), 'coverage_recall':coverage})
    return pd.DataFrame(rows)
what_if = pd.concat([heldout_box_summary(prim_boxes, 'PRIM'), heldout_box_summary(cart_boxes, 'CART')], ignore_index=True)
display(what_if.round(3))